# Cell Cycle Scoring in singlify

singlify assigns each cell a cell cycle phase (G1, S, G2/M) based on
expression of S-phase and G2/M-phase marker genes. This enables downstream
regression of cell cycle effects and identification of proliferating populations.

**Sample**: GSM3573650 (GSE125416, 75,420 cells, Homo sapiens)

In [1]:
import pandas as pd
import numpy as np

sample_dir = '/mnt/projects/debruinz_project/singlify_pipeline/quant/scrna/GSE125/GSE125416/GSM3573650'

# Load cell cycle scores from singlify output
cc = pd.read_csv(f'{sample_dir}/cell_cycle_scores.tsv', sep='\t')
print(f'Cells scored: {len(cc):,}')
print(f'\nPhase distribution:')
phase_counts = cc['phase'].value_counts()
for phase, count in phase_counts.items():
    print(f'  {phase}: {count:,} ({count/len(cc):.1%})')
print(f'\nScore statistics:')
print(f'  S-score:   mean={cc["s_score"].mean():.3f}, std={cc["s_score"].std():.3f}')
print(f'  G2M-score: mean={cc["g2m_score"].mean():.3f}, std={cc["g2m_score"].std():.3f}')

Cells scored: 75,420

Phase distribution:
  G1: 69,762 (92.5%)
  G2M: 3,805 (5.0%)
  S: 1,853 (2.5%)

Score statistics:
  S-score:   mean=0.025, std=0.070
  G2M-score: mean=0.040, std=0.113


In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Phase pie chart
colors = {'G1': '#3b82f6', 'S': '#22c55e', 'G2M': '#ef4444'}
axes[0].pie(phase_counts.values, labels=phase_counts.index, autopct='%1.1f%%',
            colors=[colors.get(p, '#6b7280') for p in phase_counts.index])
axes[0].set_title('Cell Cycle Phase Distribution')

# 2. S vs G2M score scatter
sub = cc.sample(min(5000, len(cc)), random_state=42)
scatter_colors = [colors.get(p, '#6b7280') for p in sub['phase']]
axes[1].scatter(sub['s_score'], sub['g2m_score'], c=scatter_colors, alpha=0.3, s=5)
axes[1].set_xlabel('S Score')
axes[1].set_ylabel('G2/M Score')
axes[1].set_title('S vs G2/M Score')
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.3)
axes[1].axvline(0, color='gray', linestyle='--', alpha=0.3)

# 3. Score distributions
axes[2].hist(cc['s_score'], bins=50, alpha=0.7, color=colors['S'], label='S score')
axes[2].hist(cc['g2m_score'], bins=50, alpha=0.7, color=colors['G2M'], label='G2/M score')
axes[2].set_xlabel('Score')
axes[2].set_ylabel('Cells')
axes[2].set_title('Score Distributions')
axes[2].legend()

plt.suptitle('singlify Cell Cycle Scoring — GSM3573650', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cell_cycle.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cell_cycle.png')

Saved: cell_cycle.png


In [3]:
# Combine with QC metrics to check if cycling cells differ
qc = pd.read_csv(f'{sample_dir}/cell_qc_metrics.tsv', sep='\t')
merged = cc.merge(qc, on='barcode', how='inner')

print('QC by cell cycle phase:')
print(merged.groupby('phase')[['total_umis', 'total_genes', 'mt_pct']].median().round(1).to_string())
print(f'\nInsight: Cycling cells (S, G2M) tend to have higher UMI/gene counts')
print(f'(more RNA content due to active transcription during replication)')

QC by cell cycle phase:
       total_umis  total_genes  mt_pct
phase                                 
G1          236.0        180.0     0.5
G2M        9751.0       3082.0     2.9
S          9049.0       2892.0     2.8

Insight: Cycling cells (S, G2M) tend to have higher UMI/gene counts
(more RNA content due to active transcription during replication)


## Interpretation

### Phase Assignments
- **G1** (93.9%): Non-cycling cells — quiescent or differentiated
- **G2/M** (5.1%): Cells in G2/mitosis — preparing for or undergoing division
- **S** (2.5%): Cells in S-phase — actively replicating DNA

The heavy G1 bias is expected for PBMC samples where most cells are
terminally differentiated (T cells, B cells, monocytes).

### Practical Use
1. **Regress out cell cycle**: `sc.pp.regress_out(adata, ['S_Score', 'G2M_Score'])`
2. **Identify proliferating clusters**: Clusters enriched for S/G2M phases
3. **Filter dividing cells**: For analyses requiring quiescent populations

## Summary

| Metric | Value |
|--------|-------|
| Cells scored | 75,420 |
| G1 | 69,762 (92.5%) |
| S | 1,853 (2.5%) |
| G2/M | 3,805 (5.0%) |
| Cycling (S + G2/M) | 7.5% |